# Business Analytics — Customer LTV & RFM Segmentation
## Notebook 2: RFM Analysis & Customer Segmentation

**Business Question:** Which customer segments exist in our base, and what behavior characterizes each one?

**Approach:** RFM Analysis + K-Means Clustering

**RFM Framework:**
- **R**ecency — How recently did the customer purchase? (lower = better)
- **F**requency — How often do they purchase?
- **M**onetary — How much do they spend in total?

---
**Sections:**
1. Load Data & Compute RFM
2. RFM Distribution
3. RFM Scoring (1–5)
4. Optimal Clusters (Elbow Method)
5. K-Means Segmentation
6. Segment Naming & Profiles
7. Business Conclusions

## 1. Load Data & Compute RFM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/online_retail_clean.csv', parse_dates=['InvoiceDate'])
print(f'Loaded: {len(df):,} rows, {df["CustomerID"].nunique():,} customers')
df.head(3)

In [ ]:
# Reference date: one day after the last transaction
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Reference date: {reference_date.date()}')

rfm = (
    df.groupby('CustomerID')
    .agg(
        Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('Revenue', 'sum')
    )
    .reset_index()
)

print(f'\nRFM table shape: {rfm.shape}')
print(f'\nSummary statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))

## 2. RFM Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('Recency', '#C44E52', 'Days since last purchase'),
    ('Frequency', '#4C72B0', 'Number of orders'),
    ('Monetary', '#55A868', 'Total spend (£)')
]

for ax, (col, color, xlabel) in zip(axes, configs):
    data = rfm[col]
    # Cap at 99th percentile to avoid extreme outlier distortion
    cap = data.quantile(0.99)
    ax.hist(data[data <= cap], bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.0f}')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Customers')
    ax.legend()

plt.suptitle('RFM Raw Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. RFM Scoring (1–5)

Each customer receives a score from 1 (worst) to 5 (best) on each dimension using quintiles.

- **Recency**: lower days = more recent = score 5 (reversed)
- **Frequency**: more orders = score 5
- **Monetary**: more spend = score 5

In [ ]:
def rfm_score(series, ascending=True, n=5):
    """Assign 1–n quintile score. ascending=False reverses (lower is better)."""
    # Get actual bin edges, then cut using those edges (not percentiles)
    _, bin_edges = pd.qcut(series, q=n, retbins=True, duplicates='drop')
    n_bins = len(bin_edges) - 1
    labels = list(range(1, n_bins + 1))
    if not ascending:
        labels = labels[::-1]
    return pd.cut(series, bins=bin_edges, labels=labels, include_lowest=True)

rfm['R_Score'] = rfm_score(rfm['Recency'],  ascending=False)
rfm['F_Score'] = rfm_score(rfm['Frequency'], ascending=True)
rfm['M_Score'] = rfm_score(rfm['Monetary'],  ascending=True)

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(int)
    + rfm['F_Score'].astype(int)
    + rfm['M_Score'].astype(int)
)

print('Score distribution:')
print(rfm['RFM_Score'].value_counts().sort_index())

## 4. Optimal Clusters — Elbow Method + Silhouette

In [ ]:
# Scale RFM values before clustering
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, labels, sample_size=2000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'o-', color='#4C72B0', linewidth=2, markersize=6)
axes[0].set_title('Elbow Method — Inertia vs K', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SSE)')

axes[1].plot(K_range, silhouettes, 's-', color='#DD8452', linewidth=2, markersize=6)
axes[1].set_title('Silhouette Score vs K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.suptitle('Choosing Optimal K for K-Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = K_range.start + silhouettes.index(max(silhouettes))
print(f'Best K by silhouette: {best_k} (score={max(silhouettes):.3f})')
print('Using K=4 for business interpretability (Champions / Loyal / At-Risk / Lost)')

## 5. K-Means Segmentation (K=4)

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Cluster profiles on original scale
cluster_profile = (
    rfm.groupby('Cluster')
    .agg(
        Count=('CustomerID', 'count'),
        Recency_mean=('Recency', 'mean'),
        Frequency_mean=('Frequency', 'mean'),
        Monetary_mean=('Monetary', 'mean'),
        Monetary_sum=('Monetary', 'sum')
    )
    .round(1)
)
cluster_profile['Revenue_pct'] = (cluster_profile['Monetary_sum'] / cluster_profile['Monetary_sum'].sum() * 100).round(1)
print(cluster_profile)

## 6. Segment Naming & Profiles

In [ ]:
# Map clusters to business names based on their RFM profile
# Champions: low Recency, high Frequency, high Monetary
# Loyal:     medium Recency, medium-high Frequency
# At-Risk:   high Recency (haven't bought recently), used to buy regularly
# Lost:      very high Recency, low Frequency, low Monetary

# Sort clusters by Monetary_mean to assign names consistently
sorted_clusters = cluster_profile['Monetary_mean'].rank(ascending=False).astype(int)

segment_names = {}
for cluster_id, rank in sorted_clusters.items():
    if rank == 1:
        segment_names[cluster_id] = 'Champions'
    elif rank == 2:
        segment_names[cluster_id] = 'Loyal Customers'
    elif rank == 3:
        segment_names[cluster_id] = 'At-Risk'
    else:
        segment_names[cluster_id] = 'Lost'

rfm['Segment'] = rfm['Cluster'].map(segment_names)

print('Segment assignment:')
print(rfm['Segment'].value_counts())

In [ ]:
SEGMENT_COLORS = {
    'Champions':      '#2ecc71',
    'Loyal Customers':'#3498db',
    'At-Risk':        '#e67e22',
    'Lost':           '#e74c3c'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter: Recency vs Monetary
for segment, group in rfm.groupby('Segment'):
    axes[0, 0].scatter(
        group['Recency'], group['Monetary'].clip(upper=group['Monetary'].quantile(0.99)),
        alpha=0.4, s=15, label=segment, color=SEGMENT_COLORS[segment]
    )
axes[0, 0].set_title('Recency vs Monetary by Segment', fontweight='bold')
axes[0, 0].set_xlabel('Recency (days)')
axes[0, 0].set_ylabel('Monetary (£)')
axes[0, 0].legend()

# Scatter: Frequency vs Monetary
for segment, group in rfm.groupby('Segment'):
    axes[0, 1].scatter(
        group['Frequency'], group['Monetary'].clip(upper=group['Monetary'].quantile(0.99)),
        alpha=0.4, s=15, label=segment, color=SEGMENT_COLORS[segment]
    )
axes[0, 1].set_title('Frequency vs Monetary by Segment', fontweight='bold')
axes[0, 1].set_xlabel('Frequency (orders)')
axes[0, 1].set_ylabel('Monetary (£)')
axes[0, 1].legend()

# Customer count per segment
seg_counts = rfm['Segment'].value_counts()
colors_list = [SEGMENT_COLORS[s] for s in seg_counts.index]
axes[1, 0].bar(seg_counts.index, seg_counts.values, color=colors_list, alpha=0.85)
axes[1, 0].set_title('Customer Count by Segment', fontweight='bold')
axes[1, 0].set_ylabel('Number of Customers')
for i, v in enumerate(seg_counts.values):
    axes[1, 0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Revenue per segment
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().reindex(seg_counts.index)
rev_colors = [SEGMENT_COLORS[s] for s in seg_revenue.index]
axes[1, 1].bar(seg_revenue.index, seg_revenue.values / 1000, color=rev_colors, alpha=0.85)
axes[1, 1].set_title('Total Revenue by Segment (£ thousands)', fontweight='bold')
axes[1, 1].set_ylabel('Revenue (£k)')
for i, v in enumerate(seg_revenue.values):
    axes[1, 1].text(i, v / 1000 + 5, f'£{v/1000:.0f}k', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Customer Segmentation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Snake chart — normalized RFM profile per segment
metrics = ['Recency', 'Frequency', 'Monetary']
segment_means = rfm.groupby('Segment')[metrics].mean()

# Normalize to 0-1 range for comparison
normalized = (segment_means - segment_means.min()) / (segment_means.max() - segment_means.min())
# Invert Recency so higher = better across all metrics
normalized['Recency'] = 1 - normalized['Recency']

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(metrics))

for segment in ['Champions', 'Loyal Customers', 'At-Risk', 'Lost']:
    if segment in normalized.index:
        ax.plot(x, normalized.loc[segment, metrics], 'o-',
                color=SEGMENT_COLORS[segment], linewidth=2.5, markersize=8, label=segment)

ax.set_xticks(x)
ax.set_xticklabels(['Recency\n(higher=recent)', 'Frequency\n(higher=more)', 'Monetary\n(higher=more)'])
ax.set_ylim(-0.05, 1.05)
ax.set_title('Segment Profiles — Normalized RFM', fontsize=13, fontweight='bold')
ax.set_ylabel('Normalized Score (0=worst, 1=best)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
summary = (
    rfm.groupby('Segment')
    .agg(
        Customers=('CustomerID', 'count'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean'),
        Avg_Monetary=('Monetary', 'mean'),
        Total_Revenue=('Monetary', 'sum')
    )
    .round(1)
)
summary['Revenue_%'] = (summary['Total_Revenue'] / summary['Total_Revenue'].sum() * 100).round(1)
summary['Customer_%'] = (summary['Customers'] / summary['Customers'].sum() * 100).round(1)

print('=== Segment Summary ===')
print(summary[['Customers', 'Customer_%', 'Avg_Recency', 'Avg_Frequency',
               'Avg_Monetary', 'Total_Revenue', 'Revenue_%']].to_string())

# Save for next notebook
rfm.to_csv('data/rfm_segmented.csv', index=False)
print('\nSaved: data/rfm_segmented.csv')

## 7. Business Conclusions

### Segment Profiles

| Segment | Profile | Strategic Priority |
|---------|---------|-------------------|
| **Champions** | Bought recently, buy often, spend the most | Reward & upsell — never let them churn |
| **Loyal Customers** | Buy regularly, medium spend, medium recency | Develop into Champions with targeted offers |
| **At-Risk** | Were good customers, haven't bought recently | **Highest priority** — win-back campaigns before they're lost |
| **Lost** | Haven't bought in a long time, low frequency | Re-engagement only if low cost — otherwise deprioritize |

### Key Insight: Revenue vs Customer Distribution
- Champions likely represent a small % of customers but a large % of revenue
- At-Risk customers represent the biggest opportunity: they've proven they spend, they just need re-engagement
- Losing one Champion is equivalent to acquiring 5–10 new low-value customers

### Marketing Strategy by Segment
- **Champions** → VIP program, early access, personalized recommendations
- **Loyal Customers** → Loyalty points, bundle offers, cross-sell complementary products
- **At-Risk** → Win-back email sequence: "We miss you" + time-limited discount
- **Lost** → One-shot reactivation offer; if no response, remove from active campaigns

→ **Next step:** Calculate Customer Lifetime Value per segment and quantify the revenue at risk (`03_ltv_business_impact.ipynb`)